# Jaetun suunnitteluperinnön kvantifiointi voimamuuntajakannassa PROC INBREED -menetelmällä

## Yhteenveto

Verkko-operaattorin sähköasemamuuntajat suunnitellaan peräkkäisinä sukupolvina, joista jokainen uusi malli johdetaan kahdesta edeltävästä suunnittelusta. Tässä muistikirjassa tätä suunnittelun sukupuuta käsitellään polveutumisena ja lasketaan **PROC INBREED** -menetelmällä sisäsiitos- ja additiiviset sukulaisuus- (kovarianssi-) kertoimet, jotka kvantifioivat kuinka paljon suunnitteluperintöä mitkä tahansa kaksi laitosta jakavat.

Polveutuminen on rakennettu niin, että rakenne on tulkittavissa: kaksi neljästä nykyisen kannan mallista polveutuu **sisarusparista** edeltäviä suunnitteluja ja kantaa siksi keskittynyttä perintöä, kun taas muut nojaavat erillisiin sukulinjoihin. Proseduuri palauttaa tämän täsmälleen. Kaksi sisaruksista johdettua mallia (`G3_T1`, `G3_T3`) kantavat kumpikin sisäsiitoskerrointa **F = 0,25**; kaksi muuta (`G3_T2`, `G3_T4`) ovat **F = 0**. Kannan keskimääräinen sisäsiitoskerroin on **0,0417**. Additiivisesta sukulaisuusmatriisista luettuna vähiten sukua oleva nykyisen kannan pari on **`G3_T1` & `G3_T3` (sukulaisuus = 0)** — niillä ei ole yhteistä esihistoriaa ja ne muodostavat vahvimman redundantin (N-1) parin, mikä on merkittävää, koska `G3_T1` on itse yksi sisäsiittoisimmista suunnitteluista.

## Tietolähteet

| Aineisto | Rivejä | Keskeiset muuttujat | Kuvaus |
|---------|------|---------------|-------------|
| `DesignLineage` | 12 | `Generation`, `ID`, `Parent1`, `Parent2`, `Sex` | Pieni, deterministinen sähköasemamuuntajien suunnittelun polveutuminen kolmen ei-päällekkäisen suunnittelusukupolven yli (4 perustaja-alustaa, 4 toisen sukupolven johdannaista, 4 nykyisen kannan mallia). Jokainen ei-perustajasuunnittelu tallentaa kaksi erillistä edeltävää suunnittelua (`Parent1`, `Parent2`), joista se on johdettu. `Sex` merkitsee johtavan suunnittelun roolin (M = mekaaninen/ydinlinja, F = sähköinen/käämityslinja). Polveutuminen on käsin määritelty — ei satunnainen — joten sukulaisuusrakenne tunnetaan etukäteen ja se voidaan tarkistaa proseduurin tulosteesta. |

# Jaetun suunnitteluperinnön kvantifiointi voimamuuntajakannassa

## Miksi sähköyhtiö välittää "sisäsiitoksesta"

Siirto- ja jakeluoperaattori käyttää satoja sähköaseman voimamuuntajia. Suunnittelukustannusten ja hyväksyntätyön hallitsemiseksi valmistajat suunnittelevat harvoin jokaisen muuntajan alusta alkaen — jokainen uusi malli **perii** ydingeometrian, käämitystopologian, eristysjärjestelmät ja läpivientirakenteet yhdestä tai kahdesta edeltävästä mallista. Useiden hankintakierrosten aikana tämä tuottaa aidon *suunnittelun sukupuun*: suunnittelujen polveutumisen.

Tuo jaettu perintö on piilevä luotettavuusriski. Jos kaksi samaa kuormaa suojaavaa muuntajaa (redundantti **N-1** -pari) polveutuu samasta esisuunnittelusta, piilevä suunnitteluvika — resonanssimuoto, eristyksen vanhenemismekanismi, läpiviennin ylilyöntireitti — on todennäköisesti läsnä **molemmissa** yksiköissä. Yksittäinen juurisyy voi tällöin kaataa redundantin parin samanaikaisesti: *yhteisvikaantuminen*.

**PROC INBREED** rakennettiin kvantifioimaan juuri tällaista jaettua esihistoriaa. Vaikka sen juuret ovat eläin- ja kasvinjalostuksessa, sen matematiikka — Wrightin additiivinen sukulaisuusrekursio — on toimialariippumatonta: se mittaa sen osuuden suunnitteluperinnöstä, jonka kaksi yksilöä jakavat yhteisten esi-isien kautta. Kuvaamme genetiikan sanaston laitossuunnitteluun:

| INBREED-käsite | Sähköyhtiön tulkinta |
|---|---|
| Yksilö | Muuntajan suunnittelu / malli |
| Vanhempi (isä/emo) | Edeltävä suunnittelu, josta se on johdettu |
| Sukupolvi (CLASS) | Hankinta- / suunnittelukierros |
| Sisäsiitoskerroin *F* | Itsesamankaltaisen perinnön aste suunnittelussa |
| Kovarianssi- / sukulaisuuskerroin | Jaettu suunnitteluperintö kahden suunnittelun välillä |
| Vähiten sukua oleva pari | Paras redundantti pari N-1-sietokykyä varten |

## Vaihe 1 — Määritä suunnittelun polveutuminen

Syötämme `DesignLineage`-aineiston suoraan, jotta sukulaisuusrakenne on yksiselitteinen. On kolme **ei-päällekkäistä suunnittelusukupolvea**:

- **Sukupolvi 1** — neljä perustaja-alustasuunnittelua (`G1_P1`–`G1_P4`) ilman edeltäjiä (tyhjät vanhemmat).
- **Sukupolvi 2** — neljä johdannaissuunnittelua (`G2_D1`–`G2_D4`), joista kukin on suunniteltu kahdesta **erillisestä** sukupolven 1 alustasta. `G2_D1` ja `G2_D2` ovat *täyssisaruksia* (molemmat parista `G1_P1` × `G1_P2`); `G2_D3` ja `G2_D4` ovat täyssisaruksia (molemmat parista `G1_P3` × `G1_P4`).
- **Sukupolvi 3** — neljä nykyisen kannan mallia (`G3_T1`–`G3_T4`). `G3_T1` on rakennettu sisarusparista `G2_D1` × `G2_D2`, ja `G3_T3` sisarusparista `G2_D3` × `G2_D4`; nämä kaksi suunnittelua siis keskittävät perintöä. `G3_T2` ja `G3_T4` risteyttävät kumpikin kaksi erillistä sukulinjaa.

`ID`-, `Parent1`- ja `Parent2`-sarakkeet kantavat polveutumisen; `Sex` tallentaa, mikä suunnittelulinja johti suunnittelua. Lyhyt jatko-DATA-askel tyhjentää paikkamerkin `.`, jotta perustaja-alustoilla on tyhjät vanhemmat, kuten PROC INBREED odottaa.

In [1]:
TIEDOT DesignLineage;
   PITUUS ID $12 Parent1 $12 Parent2 $12 Sex $1;
   INFILE DATALINES truncover;
   SYÖTE Generation ID $ Parent1 $ Parent2 $ Sex $;
   DATALINES;
1 G1_P1 . . M
1 G1_P2 . . M
1 G1_P3 . . M
1 G1_P4 . . F
2 G2_D1 G1_P1 G1_P2 M
2 G2_D2 G1_P1 G1_P2 F
2 G2_D3 G1_P3 G1_P4 F
2 G2_D4 G1_P3 G1_P4 M
3 G3_T1 G2_D1 G2_D2 M
3 G3_T2 G2_D1 G2_D3 F
3 G3_T3 G2_D3 G2_D4 F
3 G3_T4 G2_D2 G2_D4 M
;
SUORITA;

/* Perustaja-alustoilla ei ole edeltäjiä: tyhjennä paikkamerkkipisteet */
TIEDOT DesignLineage;
   ASETA DesignLineage;
   JOS Parent1 = '.' NIIN Parent1 = ' ';
   JOS Parent2 = '.' NIIN Parent2 = ' ';
SUORITA;

OTSIKKO 'Muuntajan suunnittelun polveutuminen';
PROSEDUURI TULOSTA TIEDOT=DesignLineage noobs NIMIKE;
   NIMIKE Generation='Sukupolvi' ID='Tunnus' Parent1='Edeltäjä 1'
         Parent2='Edeltäjä 2' Sex='Rooli';
   MUUTTUJA Generation ID Parent1 Parent2 Sex;
SUORITA;

                                          Muuntajan suunnittelun polveutuminen                                          


Sukupolvi  Tunnus    Edeltäjä 1    Edeltäjä 2  Rooli
---------  ------  ------------  ------------  -----
        1  G1_P1                               M
        1  G1_P2                               M
        1  G1_P3                               M
        1  G1_P4                               F
        2  G2_D1   G1_P1         G1_P2         M
        2  G2_D2   G1_P1         G1_P2         F
        2  G2_D3   G1_P3         G1_P4         F
        2  G2_D4   G1_P3         G1_P4         M
        3  G3_T1   G2_D1         G2_D2         M
        3  G3_T2   G2_D1         G2_D3         F
        3  G3_T3   G2_D3         G2_D4         F
        3  G3_T4   G2_D2         G2_D4         M




NOTE: DATA DesignLineage

NOTE: Processing inline DATALINES (12 lines)

NOTE: Read 12 rows from DATALINES.
NOTE: Wrote DesignLineage (12 rows, 5 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds
NOTE: DATA DesignLineage


NOTE: Read 12 rows from DesignLineage.
NOTE: Wrote DesignLineage (12 rows, 5 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds
NOTE: Option TITLE changed to Muuntajan suunnittelun polveutuminen.
NOTE: PROC PRINT data=DesignLineage

NOTE: PROC PRINT completed: 12 observations printed, 5 variables


## Vaihe 2 — Sisäsiitoskertoimet suunnittelusukupolvittain

Ajamme PROC INBREEDin **monisukupolvitilassa** nimeämällä `Generation`-muuttujan `CLASS`-lauseessa, mikä tulostaa polveutumisen yhteenvedon sukupolvittain. `VAR`-lause antaa kolme polveutumissaraketta vaaditussa järjestyksessä: yksilö, ensimmäinen edeltäjä, toinen edeltäjä.

- **IND**-optio tulostaa kunkin suunnittelun sisäsiitoskertoimen *F* — mitta siitä, kuinka keskittynyttä sen oma perintö on. Kahdesta läheisesti sukua olevasta edeltäjästä rakennettu suunnittelu kantaa korkeaa *F*:ää.
- **MATRIX**-optio tulostaa täyden additiivisen sukulaisuusmatriisin, jotta parittaisen perinnön voi lukea suoraan.
- **AVERAGE**-optio lisää kannan laajuisen keskimääräisen sisäsiitoskertoimen — yksittäisen suunnittelun monimuotoisuuden tiivistelmän.

Lähellä nollaa olevat arvot tarkoittavat riippumattomia sukulinjoja; *F* nousee, kun suunnittelun kaksi edeltäjää tulevat läheisemmin sukua toisilleen.

In [2]:
OTSIKKO 'Sisäsiitoskertoimet suunnittelusukupolvittain';
PROSEDUURI inbreed TIEDOT=DesignLineage ind average MATRIX;
   MUUTTUJA ID Parent1 Parent2;
   LUOKKA Generation;
SUORITA;

                                     Sisäsiitoskertoimet suunnittelusukupolvittain                                      


                       The INBREED Procedure                       

Dataset: DesignLineage
Number of Observations: 12

Pedigree Summary by Generation/Class
--------------------------------------
Generation 1            Members = 4
Generation 2            Members = 4
Generation 3            Members = 4

Inbreeding Coefficients (Generation 1)
--------------------------------------
ID                  F (Inbreeding)
G1_P1                 0.000000
G1_P2                 0.000000
G1_P3                 0.000000
G1_P4                 0.000000
Average Inbreeding Coefficient: 0.000000

Inbreeding Coefficients (Generation 2)
--------------------------------------
ID                  F (Inbreeding)
G2_D1                 0.000000
G2_D2                 0.000000
G2_D3                 0.000000
G2_D4                 0.000000
Average Inbreeding Coefficient: 0.000000

Inbreeding Coe


NOTE: Option TITLE changed to Sisäsiitoskertoimet suunnittelusukupolvittain.
NOTE: PROC INBREED data=DesignLineage



## Vaihe 3 — Kovarianssikertoimet tallennettuna myöhempää riskipisteytystä varten

Kun sisäsiitosnäkymä korvataan **COVAR**-optiolla, samat suhteet raportoidaan **kovarianssi- (additiivisina sukulaisuus-) kertoimina**, siinä muodossa, jonka omaisuudenhallintatiimi syöttäisi redundanssiriskin pisteytykseen. **OUTCOV=**-optio tallentaa täyden matriisin aineistoksi (`DesignCovar`), jossa sarakkeet `Col1`–`Col12` pitävät kunkin suunnittelun suhteen jokaiseen muuhun suunnitteluun (polveutumisjärjestyksessä) — valmiina liitettäväksi takaisin sähköasemien kohdennuksiin.

Tulostamme tulosaineiston, jotta tallennettu matriisi näkyy tulosteen rinnalla.

In [3]:
OTSIKKO 'Kovarianssi- (sukulaisuus-) kertoimet';
PROSEDUURI inbreed TIEDOT=DesignLineage covar MATRIX OUTCOV=DesignCovar;
   MUUTTUJA ID Parent1 Parent2;
   LUOKKA Generation;
SUORITA;

OTSIKKO 'OUTCOV=-sukulaisuusmatriisi tallennettu riskipisteytystä varten';
PROSEDUURI TULOSTA TIEDOT=DesignCovar noobs;
SUORITA;

                                         Kovarianssi- (sukulaisuus-) kertoimet                                          


                       The INBREED Procedure                       

Dataset: DesignLineage
Number of Observations: 12

Pedigree Summary by Generation/Class
--------------------------------------
Generation 1            Members = 4
Generation 2            Members = 4
Generation 3            Members = 4

Inbreeding Coefficients (Generation 1)
--------------------------------------
ID                  F (Inbreeding)
G1_P1                 0.000000
G1_P2                 0.000000
G1_P3                 0.000000
G1_P4                 0.000000

Covariance Matrix (Additive Genetic Relationship) (Generation 1)
--------------------------------------------------
ID                     G1_P1    G1_P2    G1_P3    G1_P4
G1_P1                1.0000   0.0000   0.0000   0.0000
G1_P2                0.0000   1.0000   0.0000   0.0000
G1_P3                0.0000   0.0000   1.0000   0.00


NOTE: Option TITLE changed to Kovarianssi- (sukulaisuus-) kertoimet.
NOTE: PROC INBREED data=DesignLineage

NOTE: OUTCOV= dataset DesignCovar written.
NOTE: Option TITLE changed to OUTCOV=-sukulaisuusmatriisi tallennettu riskipisteytystä varten.
NOTE: PROC PRINT data=DesignCovar

NOTE: PROC PRINT completed: 12 observations printed, 17 variables


## Vaihe 4 — Vähiten sukua olevat parit redundantteihin (N-1) asennuksiin

Tallennettu `DesignCovar`-matriisi on juuri sitä, mitä redundanssitutkimus tarvitsee. Suunnittelun *i* suhde suunnitteluun *j* sijaitsee rivillä *i*, sarakkeessa `Col`*j* (sarakkeet ovat polveutumisjärjestyksessä, joten `Col9`–`Col12` vastaavat neljää nykyisen kannan mallia `G3_T1`–`G3_T4`).

Luemme neljä nykyisen kannan riviä `DesignCovar`-aineistosta, muodostamme jokaisen järjestämättömän nykyisen kannan parin, liitämme sen sukulaisuuskertoimen ja lajittelemme vähiten sukua olevat ensin. Tavoitteena on valita redundantit parit, joiden jaettu perintö on **pienin** — nämä minimoivat mahdollisuuden, että yksi peritty suunnitteluvirhe lamauttaa molemmat puolet N-1-suojatusta positiosta.

In [4]:
/* Poimi neljä nykyisen kannan riviä; Col9..Col12 ovat suhteet
   malleihin G3_T1..G3_T4 (polveutumissarakejärjestys). Muodosta
   jokainen järjestämätön pari. */
TIEDOT Pairs;
   ASETA DesignCovar;
   MISSÄ ID SISÄLLÄ ('G3_T1','G3_T2','G3_T3','G3_T4');
   PITUUS DesignA $12 DesignB $12;
   TAULUKKO g3{4} Col9 Col10 Col11 Col12;
   TAULUKKO nm{4} $12 _temporary_
      ('G3_T1','G3_T2','G3_T3','G3_T4');
   DesignA = ID;
   TEE j = 1 ASTI 4;
      DesignB = nm{j};
      JOS DesignA < DesignB NIIN TEE;
         Relationship = g3{j};
         TULOSTE;
      LOPPU;
   LOPPU;
   SÄILYTÄ DesignA DesignB Relationship;
SUORITA;

PROSEDUURI LAJITTELE TIEDOT=Pairs; MUKAAN Relationship; SUORITA;

OTSIKKO 'Ehdokkaat redundanteiksi (N-1) pareiksi - vähiten sukua ensin';
PROSEDUURI TULOSTA TIEDOT=Pairs noobs NIMIKE;
   NIMIKE DesignA='Suunnittelu A' DesignB='Suunnittelu B'
         Relationship='Sukulaisuus';
   MUUTTUJA DesignA DesignB Relationship;
SUORITA;
OTSIKKO;

                             Ehdokkaat redundanteiksi (N-1) pareiksi - vähiten sukua ensin                              


Suunnittelu A  Suunnittelu B  Sukulaisuus
-------------  -------------  -----------
G3_T1          G3_T3                    0
G3_T2          G3_T4                 0.25
G3_T1          G3_T2                0.375
G3_T1          G3_T4                0.375
G3_T2          G3_T3                0.375
G3_T3          G3_T4                0.375




NOTE: DATA Pairs


NOTE: Read 12 rows from DesignCovar.
NOTE: Wrote Pairs (6 rows, 3 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds
NOTE: PROC SORT data=Pairs

NOTE: Unlicensed mode - input limited to 100 observations.
NOTE: Read 6 rows from Pairs.
NOTE: Wrote Pairs (6 rows, 3 columns).
NOTE: PROC SORT statement used.
NOTE: Option TITLE changed to Ehdokkaat redundanteiksi (N-1) pareiksi - vähiten sukua ensin.
NOTE: PROC PRINT data=Pairs

NOTE: PROC PRINT completed: 6 observations printed, 3 variables


## Tulosten tulkinta

**Sisäsiitoskertoimet (Vaihe 2).** Kaikki perustavat sukupolven 1 alustat ja kaikki sukupolven 2 johdannaiset näyttävät *F* = 0 — rakenteen vuoksi yhdelläkään ei ole kahta sukua olevaa edeltäjää. Kaksi sukupolven 3 mallia nousee esiin arvolla *F* = 0,25: `G3_T1` (rakennettu sisarusparista `G2_D1` × `G2_D2`) ja `G3_T3` (sisarusparista `G2_D3` × `G2_D4`). Niiden edeltäjät jäljittyvät *samaan* perustaja-alustapariin, joten niiden perintö on keskittynyt. Luotettavuuden näkökulmasta nämä ovat suunnittelut, jotka ovat alttiimpia yksittäiselle peritylle vialle, ja ne edellyttävät ylimääräistä riippumatonta hyväksyntätestausta. `G3_T2` ja `G3_T4`, jotka risteyttävät kaksi erillistä sukulinjaa, ovat *F* = 0.

**Sukulaisuusmatriisi (Vaihe 3).** Lävistäjän ulkopuoliset arvot kvantifioivat parittaisen jaetun perinnön. Kaksi sukupolven 2 sisarusparia näyttävät kumpikin sukulaisuuden 0,50 toisiinsa (`G2_D1`–`G2_D2` ja `G2_D3`–`G2_D4`), kun taas vastakkaisista sukulinjoista peräisin olevat johdannaiset ovat 0,00. Sisäsiittoiset nykyisen kannan suunnittelut kantavat itsesukulaisuutta 1,25 (= 1 + *F*), näkyvissä lävistäjällä. `DesignCovar`-aineisto tekee täyden matriisin saatavaksi liitettäväksi sähköasemien kohdennuksiin.

**Vähiten sukua olevat parit (Vaihe 4).** Kun jokainen nykyisen kannan pari järjestetään sukulaisuuden mukaan, ensimmäiseksi nousee **`G3_T1` & `G3_T3` sukulaisuudella 0,00** — ne polveutuvat täysin erillisistä esi-isä-alustoista eivätkä jaa mitään suunnitteluperintöä. Tämä on vahvin redundantti pari, ja se on erityisen arvokas, koska `G3_T1` on itse yksi kahdesta sisäsiittoisimmasta suunnittelusta: sen parittaminen täysin sukua olemattoman kumppanin kanssa suojaa sen keskittynyttä perintöä. Seuraavaksi paras pari on `G3_T2` & `G3_T4` arvolla 0,25; loput parit ovat 0,375. Kannan keskimääräinen sisäsiitoskerroin **0,0417** (tulostettu AVERAGE-optiolla vaiheessa 2) tiivistää suunnittelun kokonaismonimuotoisuuden. Hankinnan tulisi suosia pienimmän sukulaisuuden pareja N-1-positioihin ja ajan myötä laajentaa esi-isäpohjaa — laitossuunnittelun vastine geneettisen pullonkaulan välttämiselle.

**Varauma.** Tämä on havainnollistavaa synteettistä dataa; tuotantotutkimus rakentaisi polveutumisen valmistajan todellisista suunnittelun revisiotietueista ja validoisi sukulaisuuspisteet historiallisia yhteisvikaantumiskatkoja vasten ennen hankintapäätösten ohjaamista.